# Tutorial 02 – Boundary Conditions

MECHA solves a linear system of the form **W · ψ = b** where **ψ** is the water-potential vector at every node.  
The right-hand side **b** (and, in the Dirichlet case, modifications to **W**) encode the boundary conditions at the two ends of the radial water path:

| Boundary | Physical quantity | BC type |
|----------|------------------|---------|
| Soil ↔ epidermis/exodermis | soil water potential ψ_soil | **Dirichlet** (always) |
| Xylem lumen | xylem water potential ψ_xyl | **Dirichlet** (pressure-driven) |
| Xylem lumen | total xylem uptake Q | **Neumann** (flow-driven) |

**Dirichlet BC** (Case 1): fix ψ_soil and ψ_xyl → the system computes the resulting radial flow and kr.  
**Neumann BC** (Case 2): fix ψ_soil and specify the total xylem uptake Q → the system computes the resulting ψ distribution.  
Both formulations yield the same kr when the same physical state is described.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from openalea.granap.root_class import RootAnatomy
from openalea.mecha.mechaha_class import Mecha
from openalea.mecha.utils.data_loader import InData
from openalea.mecha.utils.network_builder import NetworkBuilder
from openalea.mecha.utils.visu import visualize

---
## 1. Anatomy – mature zone with fully developed xylem

We generate a synthetic root with **barrier type 3**: the endodermis is fully suberized (Casparian strip + suberin lamellae, no passage cells) and the xylem vessels are lignified.  
This represents the fully mature zone of the root where radial apoplastic transport is most restricted.

In [ ]:
root = RootAnatomy()
_ = root.export_to_adjencymatrix()

# Single maturity stage: barrier=3, height=200 µm (root cross-section depth)
config = InData()
config.geometry.set_maturity_stages([3], height=[200.0])

# Build the shared network once – both BC cases will reuse it
network = NetworkBuilder(root)
network.populate_from_network()

print(f"Xylem vessels: {len(network.cell_manager.xylem)}")
print(f"Root perimeter: {network.perimeter:.1f} µm")

In [ ]:
# Quick anatomy preview
mecha_preview = Mecha(config, network=network)
visualize(mecha_preview, visu_type="polygon")

---
## 2. Case 1 – Dirichlet boundary conditions (pressure-driven)

The classical setup: fix both ends of the water path with known potentials.

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `psi_soil_left` | 0 hPa | soil is at field capacity (no water stress) |
| `pressure_xyl_prox` | −5 000 hPa | xylem pressure imposed by transpiration demand |

### How MECHA computes Q, kr, and Kx internally

**Radial water flow Q** (`total_flow`) is the sum of flows across all soil-facing walls and junctions:

$$Q = \sum_{i \in \partial_{soil}} K_i \cdot (\psi_{soil} - \psi_i^{wall})$$

where each border-wall conductance is $K_i = k_w \cdot \frac{L_i/2 \cdot h}{d/2} \cdot 10^{-4}$, with $L_i$ the wall half-length (µm), $h$ the section height (µm), $d$ the cell-wall thickness (µm), and $k_w$ the apoplastic hydraulic conductivity (cm/hPa/d).

**Radial conductivity kr** is Q normalised by the driving force and the root surface area $A_{root} = \text{perimeter} \times h \times 10^{-4}$ cm²:

$$k_r = \frac{Q}{(\psi_{soil} - \psi_{xyl}) \cdot A_{root}}$$

**Axial conductance Kx** is computed from Poiseuille's law applied to each xylem vessel cross-section (area $A_v$ in µm²), treating all vessels in parallel:

$$K_x = \frac{\sum_v A_v^2}{8 \pi \mu} \cdot \frac{h^2}{10^{16}}$$

where $\mu = 10^{-3}$ Pa·s is water viscosity (converted to hPa·d internally). Kx therefore depends only on geometry and is identical for both BC cases.

In [ ]:
# InData defaults already set pressure_xyl_prox = -5000 and psi_soil_left = 0,
# so we just clone the config and compute.
config_dir = InData()
config_dir.geometry.set_maturity_stages([3], height=[200.0])

# boundary.scenarios is a list of scenario dicts.
#   Index 0   → the "standard" scenario used by compute_conductivities() and solve_W().
#               It defines the reference pressure state from which kr and Kx are derived.
#   Index 1+  → additional user scenarios solved in water_flux() (e.g. different soil
#               or xylem conditions for sensitivity analyses).
s = config_dir.boundary.scenarios[0]
print(f"psi_soil_left    = {s['psi_soil_left']} hPa")
print(f"pressure_xyl_prox= {s['pressure_xyl_prox']} hPa")

# Extract the BC values for later use in the tutorial.
PSI_SOIL = s['psi_soil_left']
PSI_XYL  = s['pressure_xyl_prox']

In [ ]:
mecha_dir = Mecha(config_dir, network=network)
mecha_dir.compute_conductivities()

In [ ]:
# root_hydraulic_properties is a list with one entry per maturity stage.
#   [0]         → first (here only) maturity stage (barrier=3)
#   ['kr']      → radial conductivity  (cm hPa⁻¹ d⁻¹)
#   ['Kx']      → axial conductance    (cm³ hPa⁻¹ d⁻¹)
#   ['barrier'] → barrier type used for this stage
#   ['height']  → section height h (µm)
stage_dir = mecha_dir.root_hydraulic_properties[0]
kr_dir    = stage_dir['kr']
Kx_dir    = stage_dir['Kx']

# total_flow has shape (n_maturity, n_scenarios).
#   [0][0] → maturity stage 0, scenario 0 (the standard water-flow solve)
#   Units: cm³ d⁻¹ — total radial uptake summed over all soil-facing walls/junctions
Q_dir     = mecha_dir.total_flow[0][0]

print("=== Dirichlet BC results ===")
print(f"  kr   = {kr_dir:.4e} cm hPa⁻¹ d⁻¹")
print(f"  Kx   = {Kx_dir:.4e} cm³ hPa⁻¹ d⁻¹")
print(f"  Q    = {Q_dir:.4e} cm³ d⁻¹  (total radial uptake per unit section)")
print(f"  Δψ   = {PSI_SOIL - PSI_XYL:.0f} hPa  (driving force)")

In [ ]:
visualize(mecha_dir, visu_type="water_potential",
          maturity_idx=0, scenario_idx="standard water flow",
          title="Dirichlet BC – water potential map")

---
## 3. Case 2 – Neumann boundary conditions (flow-driven)

Instead of fixing ψ_xyl, we prescribe the total radial water uptake **Q** (e.g. known from whole-plant transpiration measurements).  
MECHA distributes Q across xylem vessels proportionally to their cross-section area and finds the ψ distribution that satisfies this flux constraint.

### Implementation steps

```
1. Set mecha.psi_xyl[1, i_maturity, 0]  = NaN   → disable Dirichlet pressure BC
2. Set mecha.distributed_flow_xyl[1, 0, 0] = Q  → total xylem uptake (cm³/d)
3. Call mecha._distribute_xylem_flow()           → split Q across xylem vessels
4. Call mecha.solve_all_W()                      → solve the linear system
```

> **Note:** MECHA will print *"Error: Scenario 0 should have xylem pressure BC …"* — this is a non-fatal diagnostic.  
> It means MECHA cannot compute kr_tot during the solve; ψ_xyl is recovered afterwards as the average  
> xylem pressure from the solution, and kr can then be computed manually.

In [ ]:
# Use exactly the same Q as Case 1 so both BCs describe the same physical state
Q_target = Q_dir

config_neum = InData()
config_neum.geometry.set_maturity_stages([3], height=[200.0])

mecha_neum = Mecha(config_neum, network=network)

# --- Switch from Dirichlet to Neumann at the xylem boundary ---

# psi_xyl has shape (2, n_maturity, n_scenarios):
#   axis 0: end of root — 0 = distal (tip), 1 = proximal (base, where transpiration pull acts)
#   axis 1: maturity stage index
#   axis 2: scenario index
# Setting to NaN disables the Dirichlet pressure BC for all maturity stages in scenario 0.
mecha_neum.psi_xyl[1, :, 0] = np.nan

# distributed_flow_xyl has shape (2, n_xylem_vessels + 1, n_scenarios):
#   axis 0: end of root — 0 = distal, 1 = proximal
#   axis 1: vessel index — 0 = total flow; 1..n = per-vessel flow (filled by _distribute_xylem_flow)
#   axis 2: scenario index
# Setting index [1, 0, 0] prescribes the total proximal xylem uptake Q (cm³/d).
mecha_neum.distributed_flow_xyl[1, 0, 0] = Q_target

# Distribute Q across individual xylem vessels weighted by their cross-section area ratio
mecha_neum._distribute_xylem_flow()

print(f"Total xylem flow prescribed: {Q_target:.4e} cm³ d⁻¹")
print("Flow per vessel (first 5):")
for i in range(min(5, len(network.cell_manager.xylem))):
    print(f"  vessel {i}: {mecha_neum.distributed_flow_xyl[1, i+1, 0]:.4e} cm³ d⁻¹ (relative surface area: {mecha_neum.network.xylem_area_ratio[i]:.4e} cm²)"  )

In [ ]:
# Solve – one W system per maturity stage
# Expected diagnostic: "Error: Scenario 0 should have xylem pressure BC ..."
# This is non-fatal: MECHA recovers ψ_xyl from the solution average.
mecha_neum.solve_all_W()

In [ ]:
# After solve_all_W(), psi_xyl[1][0][0] is filled with the average xylem pressure
psi_xyl_neum = mecha_neum.psi_xyl[1][0][0]
Q_neum       = mecha_neum.total_flow[0][0]
height       = mecha_neum.geometry.maturity_stages[0]['height']   # µm
perimeter    = mecha_neum.network.perimeter                        # µm

# Manually reproduce the kr formula (same as standard_water_flow in mecha_class.py)
kr_neum = Q_neum / (PSI_SOIL - psi_xyl_neum) / perimeter / height / 1.0E-04

# Kx is geometry-only (Poiseuille) – compute it the same way
_, Kx_neum = mecha_neum.calculate_axial_conductance(i_maturity=0)

print("=== Neumann BC results ===")
print(f"  ψ_xyl (recovered) = {psi_xyl_neum:.1f} hPa")
print(f"  kr                = {kr_neum:.4e} cm hPa⁻¹ d⁻¹")
print(f"  Kx                = {Kx_neum:.4e} cm³ hPa⁻¹ d⁻¹")
print(f"  Q (soil uptake)   = {Q_neum:.4e} cm³ d⁻¹")

In [ ]:
visualize(mecha_neum, visu_type="water_potential",
          maturity_idx=0, scenario_idx="standard water flow",
          title="Neumann BC – water potential map")

---
## 4. Comparison

When both cases describe the same physical state (same Q, same ψ_soil), they must produce the same kr, Kx and ψ_xyl.
Any difference reflects numerical precision only.

In [ ]:
psi_xyl_dir = mecha_dir.psi_xyl[1][0][0]  # −5000 hPa (prescribed)

print(f"{'Property':<28} {'Dirichlet':>14} {'Neumann':>14} {'Δ (%)':>10}")
print("-" * 70)
rows = [
    ("ψ_xyl (hPa)",   psi_xyl_dir,  psi_xyl_neum),
    ("Q (cm³/d)",     Q_dir,        Q_neum),
    ("kr (cm/hPa/d)", kr_dir,       kr_neum),
    ("Kx (cm³/hPa/d)",Kx_dir,       Kx_neum),
]
for label, v_dir, v_neum in rows:
    rel = abs(v_dir - v_neum) / abs(v_dir) * 100 if v_dir != 0 else float('nan')
    print(f"{label:<28} {v_dir:>14.4e} {v_neum:>14.4e} {rel:>9.2f}%")

---
## Summary

| BC type | When to use | What you fix | What you get |
|---------|-------------|-------------|-------------|
| **Dirichlet** | ψ_xyl measured or imposed | ψ_soil and ψ_xyl | Q, kr, Kx |
| **Neumann** | transpiration rate known | ψ_soil and Q | ψ_xyl, kr, Kx |

Both formulations give identical kr and Kx; the choice is dictated by which quantity is experimentally accessible.